# NYC 311 Noise Complaints: API Download

**Project:** Does the Full Moon Bring More Chaos to New York City?  
**Dataset:** NYC 311 Service Requests from 2020 to Present (`erm2-nwe9`)

This notebook pulls 311 service requests from the NYC Open Data API. The default query keeps complaint types that begin with `Noise`, which gives us a focused starting point for studying nighttime disturbances. The notebook downloads records in pages and saves them to a CSV file.

**Important:** The `/api/v3/` query endpoint is SODA3. Filtered query requests require a Socrata app token. Register for one here: https://dev.socrata.com/register. The token is entered privately in the next section and is not saved in the notebook.

API documentation: https://support.socrata.com/hc/en-us/articles/43491231777047-Using-the-SODA3-API

## 1. Install and import packages

Run this cell first. `requests` sends the API calls and `pandas` handles the returned data.

In [16]:
!pip -q install requests pandas

import os
import time
from datetime import date, timedelta
from getpass import getpass
from pathlib import Path

import pandas as pd
import requests
from IPython.display import display

pd.set_option("display.max_columns", 60)
pd.set_option("display.max_colwidth", 100)

## 2. Connect to the NYC Open Data API

The API token is requested with a hidden input so it will not appear in the notebook output. Do not paste your token into a code cell or share it in GitHub.

In [17]:
import requests
import pandas as pd
import time
import os

API_URL = "https://data.cityofnewyork.us/api/v3/views/erm2-nwe9/query.json"

APP_TOKEN = "P0FHkNgGQ6QB7I8MH8cVkwFAb"

headers = {
    "X-App-Token": APP_TOKEN,
    "Accept": "application/json",
    "Content-Type": "application/json"
}

session = requests.Session()
session.headers.update(headers)

# Test API connection
test_payload = {
    "query": "SELECT unique_key, created_date, complaint_type LIMIT 5",
    "page": {
        "pageNumber": 1,
        "pageSize": 5
    }
}

response = session.post(API_URL, json=test_payload, timeout=60)

print("Status:", response.status_code)

if response.status_code == 200:
    display(pd.DataFrame(response.json()))
else:
    print(response.text[:2000])
    response.raise_for_status()

Status: 200


,unique_key,created_date,complaint_type,:id,:version,:created_at,:updated_at,:@computed_region_f5dn_yrer,:@computed_region_yeji_bk3q,:@computed_region_sbqj_enih,:@computed_region_92fq_4b7q
0,71044266,2026-11-09T02:06:35.000,Noise,row-es72.sn6y.uu9d,rv-7ykc_4jqr_b2sv,2026-11-10T02:32:48.798Z,2026-11-10T02:37:10.345Z,14,2,50,27
1,71044500,2026-11-09T02:06:32.000,Noise - Residential,row-jii6~fybu-quwp,rv-y7b8~96h8~gebk,2026-11-10T02:32:48.798Z,2026-11-10T02:37:10.345Z,45,2,47,25
2,71050770,2026-11-09T02:06:24.000,For Hire Vehicle Complaint,row-xibq_n5r7.c4zj,rv-r822_n7gc~uwzt,2026-11-10T02:32:48.798Z,2026-11-10T02:37:11.932Z,56,4,1,32
3,71050000,2026-11-09T02:06:12.000,Noise - Residential,row-7xux_z5fd~erp8,rv-b76u-r9wb-ffrq,2026-11-10T02:32:48.798Z,2026-11-10T02:37:11.932Z,22,3,67,3
4,71045758,2026-11-09T02:04:37.000,Illegal Parking,row-csfc.6i8r~ki5w,rv-v9a5~gjey~h2sm,2026-11-10T02:32:48.798Z,2026-11-10T02:37:10.345Z,9,2,45,7


## 3. Choose the data pull settings

The default date range starts in 2020 and runs through today. The first run is set to **test mode**, which downloads only one page (up to 1,000 records). Check that the request works and that the records look right before downloading the full result.

When you are ready for the full download, change `TEST_MODE` to `False` and run the download cell again. The full noise-only pull may contain a large number of records and take some time.

In [18]:
# Date range. END_DATE is exclusive, so tomorrow at midnight includes all of today.
START_DATE = "2020-01-01T00:00:00"
END_DATE = (date.today() + timedelta(days=1)).strftime("%Y-%m-%dT00:00:00")

# Keep True for the first test. Change to False to retrieve every matching page.
TEST_MODE = False
MAX_PAGES = 1 if TEST_MODE else None

# True keeps only complaint types beginning with "Noise".
# Set False only if you intentionally want the much larger full 311 dataset.
NOISE_ONLY = True

PAGE_SIZE = 1000
MAX_RETRIES = 6
REQUEST_DELAY_SECONDS = 0.1
REQUEST_TIMEOUT_SECONDS = 660

OUTPUT_CSV = "/content/nyc_311_noise_complaints.csv"
RESET_OUTPUT = True  # True replaces an older CSV when a new download starts.

print("Start:", START_DATE)
print("End (exclusive):", END_DATE)
print("Noise only:", NOISE_ONLY)
print("Test mode:", TEST_MODE)
print("Output:", OUTPUT_CSV)

Start: 2020-01-01T00:00:00
End (exclusive): 2026-11-11T00:00:00
Noise only: True
Test mode: False
Output: /content/nyc_311_noise_complaints.csv


## 4. Build the SoQL query

The selected columns below match the fields in the query you supplied. The query is sent as a POST request, which is recommended for SODA3 because it handles longer queries more reliably.

In [19]:
columns = """
unique_key,
created_date,
closed_date,
agency,
agency_name,
complaint_type,
descriptor,
descriptor_2,
location_type,
incident_zip,
incident_address,
street_name,
cross_street_1,
cross_street_2,
intersection_street_1,
intersection_street_2,
address_type,
city,
landmark,
facility_type,
status,
due_date,
resolution_description,
resolution_action_updated_date,
community_board,
council_district,
police_precinct,
bbl,
borough,
x_coordinate_state_plane,
y_coordinate_state_plane,
open_data_channel_type,
park_facility_name,
park_borough,
vehicle_type,
taxi_company_borough,
taxi_pick_up_location,
bridge_highway_name,
bridge_highway_direction,
road_ramp,
bridge_highway_segment,
latitude,
longitude,
location
"""

where_parts = [
    f"created_date >= '{START_DATE}'",
    f"created_date < '{END_DATE}'"
]

if NOISE_ONLY:
    where_parts.append("complaint_type LIKE 'Noise%'")

where_clause = " AND ".join(where_parts)
SOQL_QUERY = f"SELECT {columns} WHERE {where_clause} ORDER BY created_date ASC, unique_key ASC"
COUNT_QUERY = f"SELECT COUNT(*) AS total_records WHERE {where_clause}"

print("Query ready:")
print(SOQL_QUERY)

Query ready:
SELECT 
unique_key,
created_date,
closed_date,
agency,
agency_name,
complaint_type,
descriptor,
descriptor_2,
location_type,
incident_zip,
incident_address,
street_name,
cross_street_1,
cross_street_2,
intersection_street_1,
intersection_street_2,
address_type,
city,
landmark,
facility_type,
status,
due_date,
resolution_description,
resolution_action_updated_date,
community_board,
council_district,
police_precinct,
bbl,
borough,
x_coordinate_state_plane,
y_coordinate_state_plane,
open_data_channel_type,
park_facility_name,
park_borough,
vehicle_type,
taxi_company_borough,
taxi_pick_up_location,
bridge_highway_name,
bridge_highway_direction,
road_ramp,
bridge_highway_segment,
latitude,
longitude,
location
 WHERE created_date >= '2020-01-01T00:00:00' AND created_date < '2026-11-11T00:00:00' AND complaint_type LIKE 'Noise%' ORDER BY created_date ASC, unique_key ASC


## 5. Check how many records match the query

This count request does not download the rows. It gives us an estimate of the size of the pull before we start.

In [20]:
count_payload = {
    "query": COUNT_QUERY,
    "includeSynthetic": False,
    "timeout": 600
}

count_response = session.post(
    API_URL,
    json=count_payload,
    timeout=REQUEST_TIMEOUT_SECONDS
)

if not count_response.ok:
    print(count_response.text[:2000])
count_response.raise_for_status()

count_result = count_response.json()
count_df = pd.DataFrame(count_result if isinstance(count_result, list) else count_result.get("data", []))
display(count_df)

,total_records
0,5246994


## 6. Download the records, 1,000 at a time

SODA3 uses `pageNumber` and `pageSize`. Page numbers start at 1. This loop requests each page, appends it to the CSV, and stops when the API returns an empty page or a page with fewer than 1,000 rows.

If the request fails, the error response is printed so it is easier to diagnose an invalid token, query, or API issue.

In [ ]:
def request_page(page_number):
    """Request one page from the SODA3 query endpoint, with retries."""
    payload = {
        "query": SOQL_QUERY,
        "page": {
            "pageNumber": page_number,
            "pageSize": PAGE_SIZE
        },
        "includeSynthetic": False,
        "timeout": 600
    }

    transient_statuses = {429, 500, 502, 503, 504}

    for attempt in range(MAX_RETRIES):
        try:
            response = session.post(
                API_URL,
                json=payload,
                timeout=REQUEST_TIMEOUT_SECONDS
            )
        except requests.exceptions.RequestException as exc:
            if attempt == MAX_RETRIES - 1:
                raise
            wait_seconds = min(60, 2 ** attempt)
            print(f"Connection problem: {exc}. Retrying in {wait_seconds}s...")
            time.sleep(wait_seconds)
            continue

        if response.status_code in transient_statuses:
            retry_after = response.headers.get("Retry-After")
            wait_seconds = int(retry_after) if retry_after and retry_after.isdigit() else min(60, 2 ** attempt)
            if attempt == MAX_RETRIES - 1:
                print(response.text[:2000])
                response.raise_for_status()
            print(f"API returned {response.status_code}. Retrying in {wait_seconds}s...")
            time.sleep(wait_seconds)
            continue

        if not response.ok:
            print("API error response:")
            print(response.text[:3000])
        response.raise_for_status()

        result = response.json()
        if isinstance(result, list):
            return result
        if isinstance(result, dict):
            # Handles wrapped responses if the API returns one.
            for key in ("data", "results", "rows"):
                if key in result and isinstance(result[key], list):
                    return result[key]
        raise ValueError(f"Unexpected API response format: {type(result)}")

    raise RuntimeError("Request failed after all retry attempts.")


# Start with a clean file if RESET_OUTPUT is True.
if RESET_OUTPUT and os.path.exists(OUTPUT_CSV):
    os.remove(OUTPUT_CSV)

page_number = 1
total_rows = 0

while True:
    records = request_page(page_number)

    if not records:
        print("The API returned an empty page. Download finished.")
        break

    page_df = pd.DataFrame(records)
    page_df.to_csv(
        OUTPUT_CSV,
        mode="a",
        header=(not os.path.exists(OUTPUT_CSV)),
        index=False
    )

    total_rows += len(page_df)
    print(f"Page {page_number}: {len(page_df):,} rows | Total: {total_rows:,}")

    if MAX_PAGES is not None and page_number >= MAX_PAGES:
        print("Stopped because TEST_MODE / MAX_PAGES is enabled.")
        break

    if len(records) < PAGE_SIZE:
        print("The last page contained fewer than PAGE_SIZE rows. Download finished.")
        break

    page_number += 1
    time.sleep(REQUEST_DELAY_SECONDS)

print(f"\nDownloaded {total_rows:,} rows.")
print("Saved to:", OUTPUT_CSV)
print("File size (MB):", round(os.path.getsize(OUTPUT_CSV) / (1024 ** 2), 2))

Page 1: 1,000 rows | Total: 1,000
Page 2: 1,000 rows | Total: 2,000
Page 3: 1,000 rows | Total: 3,000
Page 4: 1,000 rows | Total: 4,000
Page 5: 1,000 rows | Total: 5,000
Page 6: 1,000 rows | Total: 6,000
Page 7: 1,000 rows | Total: 7,000
Page 8: 1,000 rows | Total: 8,000
Page 9: 1,000 rows | Total: 9,000
Page 10: 1,000 rows | Total: 10,000
Page 11: 1,000 rows | Total: 11,000
Page 12: 1,000 rows | Total: 12,000
Page 13: 1,000 rows | Total: 13,000
Page 14: 1,000 rows | Total: 14,000
Page 15: 1,000 rows | Total: 15,000
Page 16: 1,000 rows | Total: 16,000
Page 17: 1,000 rows | Total: 17,000
Page 18: 1,000 rows | Total: 18,000
Page 19: 1,000 rows | Total: 19,000
Page 20: 1,000 rows | Total: 20,000
Page 21: 1,000 rows | Total: 21,000
Page 22: 1,000 rows | Total: 22,000
Page 23: 1,000 rows | Total: 23,000
Page 24: 1,000 rows | Total: 24,000
Page 25: 1,000 rows | Total: 25,000
Page 26: 1,000 rows | Total: 26,000
Page 27: 1,000 rows | Total: 27,000
Page 28: 1,000 rows | Total: 28,000
Page 29: 1

## 7. Load and inspect the downloaded CSV

If `TEST_MODE` is still `True`, this will show only the first 1,000 records. After checking the result, set `TEST_MODE = False`, rerun the download cell, and then rerun this section to load the full pull.

In [ ]:
if not os.path.exists(OUTPUT_CSV):
    raise FileNotFoundError("The CSV was not created. Run the download section first.")

df = pd.read_csv(OUTPUT_CSV, low_memory=False)

# Parse the timestamp so we can inspect dates and hours.
df["created_date"] = pd.to_datetime(df["created_date"], errors="coerce")
if "closed_date" in df.columns:
    df["closed_date"] = pd.to_datetime(df["closed_date"], errors="coerce")

print("Dataset dimensions:", df.shape)
print("Earliest request:", df["created_date"].min())
print("Latest request:", df["created_date"].max())
display(df.head())

## 8. Check complaint categories and missing values

This is a quick check that the filter returned the complaint types we expected and that the main fields are usable.

In [ ]:
print("Most common complaint types:")
display(df["complaint_type"].value_counts(dropna=False).head(20))

print("\nMissing values in key columns:")
key_columns = [c for c in ["unique_key", "created_date", "complaint_type", "descriptor", "borough", "latitude", "longitude"] if c in df.columns]
display(df[key_columns].isna().sum().sort_values(ascending=False).to_frame("missing_count"))

## 9. Create a simple nighttime complaint count

For this project, define nighttime as **8:00 p.m. through 3:59 a.m.** The early-morning hours are assigned to the previous evening, so one night runs from 8 p.m. to 4 a.m.

This cell is useful after the full download. If you are still in test mode, the counts describe only the small test sample and should not be used for analysis.

In [ ]:
night_start_hour = 20
early_morning_end_hour = 4

night_data = df.dropna(subset=["created_date"]).copy()
night_data["hour"] = night_data["created_date"].dt.hour
night_data["is_night"] = (
    (night_data["hour"] >= night_start_hour) |
    (night_data["hour"] < early_morning_end_hour)
)

night_data = night_data.loc[night_data["is_night"]].copy()
night_data["night_date"] = night_data["created_date"].dt.normalize()
night_data.loc[
    night_data["hour"] < early_morning_end_hour, "night_date"
] -= pd.Timedelta(days=1)

nightly_counts = (
    night_data.groupby(night_data["night_date"].dt.date)
    .size()
    .rename("night_noise_complaints")
    .reset_index(name="night_noise_complaints")
    .rename(columns={"night_date": "date"})
)

display(nightly_counts.head())
print("Number of nights represented:", len(nightly_counts))
print("Average nighttime noise complaints:", round(nightly_counts["night_noise_complaints"].mean(), 2))

## Notes for the project

- This pull contains **311 noise complaints**, not every possible measure of city activity. Dog bites and NYPD crime incidents come from separate NYC Open Data datasets and will need separate API pulls.
- A 311 complaint is a report made by a member of the public. It is not proof that a disturbance or crime occurred.
- The date and hour analysis should use New York local time. Check the timestamp format and dataset metadata before making final nighttime classifications.
- For the classification project, the team can aggregate complaints by night and label unusually high-complaint nights as `1` and other nights as `0`. The threshold should be set using the training data only to avoid data leakage.
- Keep the original row-level CSV. It is the raw source data for cleaning and reproducibility.

**Dataset page:** https://data.cityofnewyork.us/Social-Services/311-Service-Requests-from-2020-to-Present/erm2-nwe9  
**Socrata SODA3 documentation:** https://support.socrata.com/hc/en-us/articles/43491231777047-Using-the-SODA3-API  
**Socrata app token registration:** https://dev.socrata.com/register